# Practice: Choose the Cheapest Adequate Method

**Computational Text Analysis Pack · Module 28 — The Neural Critique**

*Technique:* Applying the six-step decision procedure to real questions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/15-the-neural-critique/28c_practice_method_decision.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **For each of five real research questions, what is the cheapest method that
> actually answers it?**

This is the course's closing exercise. Module 28b gave the six-step ladder; this
walks concrete questions down it and shows most of them stopping early.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
LADDER = ["1 count it", "2 a rule", "3 classical supervised",
          "4 frozen embeddings", "5 fine-tune", "6 generate"]
QUESTIONS = [
 ("Which words distinguish health from political coverage?",
  1, "Keyness with effect size and dispersion (M4). No model needed at all."),
 ("Find every mention of a named commission in an archive.",
  2, "Gazetteer plus pattern rules (M15c). Auditable, no training data."),
 ("Sort 20,000 articles into seven topics, 300 labels available.",
  3, "Logistic regression on TF-IDF (M7, M26c). Fine-tuning would overfit."),
 ("Find articles about the same event across 16 languages.",
  4, "Cross-lingual embeddings (M23b). Nothing cheaper crosses languages."),
 ("Tag named entities in Yoruba to research-grade accuracy.",
  5, "Fine-tune on MasakhaNER (M23c) — and expect the diacritic problem."),
]
print(f"{'question':<58}{'stops at':>10}")
print("-" * 70)
for q, step, why in QUESTIONS:
    print(f"{q:<58}{LADDER[step-1]:>10}")
print()
for q, step, why in QUESTIONS:
    print(f"Q: {q}\n   -> {why}\n")

In [ ]:
import numpy as np
counts = Counter(step for _, step, _ in QUESTIONS)
plt.figure(figsize=(7.5, 3.6))
plt.bar([LADDER[i-1] for i in range(1, 7)],
        [counts.get(i, 0) for i in range(1, 7)], color="steelblue")
plt.ylabel("questions"); plt.xticks(rotation=20, ha="right", fontsize=8)
plt.title("Where real questions stop on the ladder")
plt.tight_layout()
print(f"{sum(counts.get(i,0) for i in (1,2))} of {len(QUESTIONS)} stop before any")
print("machine learning at all. In a real research programme the proportion is")
print("higher, not lower.")

In [ ]:
# Prove step 1 answers question 1, so the claim is not just an assertion.
ca, cb = Counter(), Counter()
for t in df[df["category"] == "health"]["text"]: ca.update(tokenise(str(t)))
for t in df[df["category"] == "politics"]["text"]: cb.update(tokenise(str(t)))
na, nb = sum(ca.values()), sum(cb.values())
rows = [{"word": w, "log_ratio": np.log2(((ca[w]+.5)/na)/((cb[w]+.5)/nb)),
         "total": ca[w]+cb[w]} for w in ca if ca[w]+cb[w] >= 60]
k = pd.DataFrame(rows).sort_values("log_ratio", ascending=False)
print("Question 1, answered by counting (seconds, no model, fully interpretable):\n")
print("health-leaning :", ", ".join(k.head(10)["word"]))
print("politics-leaning:", ", ".join(k.tail(10)["word"][::-1]))
print()
print("A transformer would also answer this, at hundreds of times the cost, and")
print("could not show you the list.")

## The closing argument of this course

Twenty-eight modules, and the ladder puts most questions on the bottom two
rungs. That is not an anticlimax; it is the finding.

**What the classic ecosystem gives you:** interpretability, reproducibility, low
cost, and results a humanities reader can audit word by word. Modules 1–22.

**What the neural ecosystem gives you:** transfer across languages, tolerance of
scarce labels, and accuracy on tasks where surface form fails. Modules 23–28.

**What neither gives you:** a research question, a defensible corpus, or the
judgement to know what a number means. Those are modules 1, 11 and 22 — the ones
with the least code.

**The sentence worth carrying out of the course:** *report the cheapest method
that answered your question, and say what you tried that did not work.* It is
better scholarship than any accuracy figure, and it is the thing almost nobody
does.

### 🔧 Your turn 1

Take three questions from your own research and place each on the ladder. Then check: did you already reach for a method further down than necessary?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Most people find at least one, and the reason is rarely ignorance — it is that
the more elaborate method is easier to publish and easier to describe as
innovative. Naming that pressure is part of the methodological literacy this
course is for. The counter-move is concrete: report the simpler baseline
alongside, and if it wins, say so. A paper that shows a transformer failing to
beat keyness on a humanities question has told the field something genuinely
useful.

</details>